# 贝叶斯网络分析

本Notebook使用贝叶斯网络和Apriori算法进行高风险组合分析。

## 步骤
1. 加载已离散化的数据集
2. 确保所有列转换为category类型
3. 学习贝叶斯网络结构和参数
4. 生成高风险条件组合


In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# 添加src目录到路径，以便导入模块
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"已添加src路径: {src_path}")


In [ ]:
# 加载已离散化的数据集（使用清洗后的数据）
data_path = r"D:\railway_risk_analysis\src\new\df_california_selected_raw_cleaned.csv"

print("=" * 80)
print("加载数据集")
print("=" * 80)
print(f"数据路径: {data_path}")

df = pd.read_csv(data_path)
print(f"\n原始数据形状: {df.shape}")
print(f"原始列数: {len(df.columns)}")
print(f"\n注意: 使用的是清洗后的数据集，包含 TrackDensity_clean 和 TrackDensity_BINS 列")
print(f"\n前5行预览:")
print(df.head())


In [ ]:
# 确保所有列都转换为category类型（pgmpy需要）
print("=" * 80)
print("转换数据类型为category")
print("=" * 80)

# 排除不需要转换的列（ID列等）
exclude_cols = [
    'AccidentId', 'ReportYear', 'Accident Number', 
    'Report Number', 'ID', 'id', 'index'
]

# 找出实际存在的排除列
existing_exclude = [col for col in exclude_cols if col in df.columns]
if existing_exclude:
    print(f"排除列: {existing_exclude}")

# 转换所有非排除列为category类型
converted_cols = []
for col in df.columns:
    if col not in existing_exclude:
        if df[col].dtype.name != 'category':
            df[col] = df[col].astype('category')
            converted_cols.append(col)

print(f"\n转换了 {len(converted_cols)} 列为category类型")
if len(converted_cols) > 0:
    print(f"前10个转换的列: {converted_cols[:10]}")

# 检查数据类型
print(f"\n数据类型统计:")
print(df.dtypes.value_counts())

print(f"\n最终数据形状: {df.shape}")
print(f"category类型列数: {sum(df.dtypes == 'category')}")


In [ ]:
# 导入贝叶斯网络分析函数
from association_miner import learn_bayesian_network, generate_high_risk_combinations

print("成功导入函数:")
print(f"  - learn_bayesian_network")
print(f"  - generate_high_risk_combinations")


In [ ]:
# 步骤1: 学习贝叶斯网络
print("=" * 80)
print("步骤1: 学习贝叶斯网络")
print("=" * 80)

# 创建结果保存目录
output_dir = project_root / 'results'
output_dir.mkdir(exist_ok=True)
print(f"结果保存目录: {output_dir}")

# 调用learn_bayesian_network
model = learn_bayesian_network(
    df=df,
    target_col="Accident Type",
    output_dir=str(output_dir),
    exclude_cols=existing_exclude if existing_exclude else None
)

print("\n" + "=" * 80)
print("贝叶斯网络学习完成！")
print("=" * 80)
print(f"模型类型: {type(model)}")
print(f"模型节点数: {len(model.nodes())}")
print(f"模型边数: {len(model.edges())}")


In [ ]:
# 步骤2: 生成高风险组合
print("=" * 80)
print("步骤2: 生成高风险组合")
print("=" * 80)

# Apriori参数
min_support = 0.01  # 最小支持度阈值
max_len_itemset = 3  # 项集的最大长度

print(f"Apriori参数:")
print(f"  - min_support: {min_support}")
print(f"  - max_len_itemset: {max_len_itemset}")

# 调用generate_high_risk_combinations
output_path = str(output_dir / 'high_risk_combinations_analysis_from_preprocessed.csv')

results_df = generate_high_risk_combinations(
    df=df,
    model=model,
    target_col="Accident Type",
    min_support=min_support,
    max_len=max_len_itemset,
    output_path=output_path,
    exclude_cols=existing_exclude if existing_exclude else None
)

print("\n" + "=" * 80)
print("高风险组合生成完成！")
print("=" * 80)


In [ ]:
# 显示结果预览
print("=" * 80)
print("结果预览")
print("=" * 80)

if results_df is not None and len(results_df) > 0:
    print(f"\n总记录数: {len(results_df)}")
    print(f"\n结果列: {list(results_df.columns)}")
    print(f"\n前10条高风险组合（按uplift降序）:")
    print(results_df.head(10).to_string())
    
    print(f"\n统计信息:")
    print(f"  - 平均支持度: {results_df['support_rate'].mean():.6f}")
    print(f"  - 平均概率: {results_df['prob'].mean():.6f}")
    print(f"  - 平均提升度: {results_df['uplift'].mean():.6f}")
    print(f"  - 最高提升度: {results_df['uplift'].max():.6f}")
    print(f"  - 最高概率: {results_df['prob'].max():.6f}")
    
    print(f"\n结果已保存到: {output_path}")
else:
    print("警告: 未生成任何结果")


## 分析完成

- 贝叶斯网络结构图已保存到 `results/bayesian_network_structure.png`
- 高风险组合结果已保存到 `results/high_risk_combinations_analysis_from_preprocessed.csv`
